# Import Libraries

Enable automatic reloading of modules during interactive development:

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit # type: ignore

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling3_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
)

In [2]:
# from sklearn.linear_model import LogisticRegression

estimator = StatsModelsEstimator(Logit)
# estimator = LogisticRegression()

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    features_to_drop = (
        'Re', 
        'We', 
        'init_volume_fraction',
        'particle_droplet_diameter_ratio', 
        'sedimentation_Re',
        # 'particle_liquid_density_ratio',
        'sedimentation_Stk'
        # 'sign_sedimentation_Re',
        # 'volume_fraction', 
        # 'relative_roughness', 
        # 'inclination',
        # 'wettability',
    ),
    model_postfix='base',
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


None

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sign_sedimentation_Re',
                                  'particle_liquid_density_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimization terminated successfully.
         Current function value: 0.404380
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.376612
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.397960
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.389046
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.401565
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.371844
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.387222
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.384420
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  297
Model:                    

,0
dataset,df_dimless
target,splashing
model,Logit_splashing_base
params,{'estimator': StatsModelsEstimator(model_class...
holdout_test_accuracy,0.826667
...,...
cv_train_recall_std,0.009788
cv_train_roc_auc,"[0.8876537880423052, 0.9035515618314078, 0.892..."
cv_train_roc_auc_mean,0.896184
cv_train_roc_auc_median,0.895975


Model saved in ../results/models_modelling3/Logit_splashing_base


In [3]:
model = joblib.load(
    '../results/models_modelling3/Logit_splashing_base'
)
model.steps[-1][-1].results_.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      y   No. Observations:                  297
Model:                          Logit   Df Residuals:                      290
Method:                           MLE   Df Model:                            6
Date:                Wed, 09 Oct 2024   Pseudo R-squ.:                  0.4082
Time:                        22:33:35   Log-Likelihood:                -114.17
converged:                       True   LL-Null:                       -192.93
Covariance Type:            nonrobust   LLR p-value:                 1.983e-31
=================================================================================================
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
inclination                       2.9317      0.637      4.602      0.000       1.683       4.180
volume_fraction                   3.2403      0.861      3.764      0.000       1.553       4.928
sign_sedimentation_Re             0.6854      0.204      3.364      0.001       0.286       1.085
particle_liquid_density_ratio    -0.3194      0.139     -2.300      0.021      -0.591      -0.047
relative_roughness                0.3266      0.167      1.954      0.051      -0.001       0.654
K                                 3.7320      0.503      7.422      0.000       2.747       4.717
wettability                      -0.5327      0.211     -2.527      0.011      -0.946      -0.120
=================================================================================================
"""

In [4]:
model.predict_proba(
    ml_pipe.get_X_y(ml_pipe.test)[0]
)[:5,:]

array([[0.03358321, 0.96641679],
       [0.04513453, 0.95486547],
       [0.02539107, 0.97460893],
       [0.90765839, 0.09234161],
       [0.00162037, 0.99837963]])